In [0]:
%sql
-- ===============================================================================
-- Creating the Silver layer tables out of the Bronze tables
-- Check / convert to correct units and calculate null percentages per system
-- ===============================================================================

CREATE SCHEMA IF NOT EXISTS pvdaq_catalog.silver;

USE CATALOG pvdaq_catalog;
USE SCHEMA silver;
SELECT current_catalog(), current_schema();

CREATE TABLE IF NOT EXISTS inverters AS SELECT * FROM pvdaq_catalog.bronze.inverters;
CREATE TABLE IF NOT EXISTS meters AS SELECT * FROM pvdaq_catalog.bronze.meters;
CREATE TABLE IF NOT EXISTS metrics AS SELECT * FROM pvdaq_catalog.bronze.metrics;
CREATE TABLE IF NOT EXISTS modules AS SELECT * FROM pvdaq_catalog.bronze.modules;
CREATE TABLE IF NOT EXISTS mount AS SELECT * FROM pvdaq_catalog.bronze.mount;
CREATE TABLE IF NOT EXISTS other_instruments AS SELECT * FROM pvdaq_catalog.bronze.other_instruments;
CREATE TABLE IF NOT EXISTS pvdata_2020_sample AS SELECT * FROM pvdaq_catalog.bronze.pvdata_2020_sample;
CREATE TABLE IF NOT EXISTS site AS SELECT * FROM pvdaq_catalog.bronze.site;
CREATE TABLE IF NOT EXISTS system AS SELECT * FROM pvdaq_catalog.bronze.system;

-- check the AC power units
SELECT DISTINCT units FROM metrics
    WHERE common_name = "AC power";
-- units of metrics with common_name AC power: W, kWh. kW, kVARh

-- check the POA irradiance units
SELECT DISTINCT units FROM metrics 
    WHERE common_name = "Irradiance POA";
-- units of metrics with common_name Irradiance POA: W/m^2, mV

-- check the ambient temperature units
SELECT DISTINCT units from metrics
    WHERE common_name = "Temperature ambient";
-- units of metrics with common_name Temperature ambient: C

-- check the wind speed units
SELECT DISTINCT units from metrics
    WHERE common_name = "Wind speed";
-- units of metrics with common_name Wind speed: m/s

/*
Units required in PVAnalytics function performance_ratio_nrel():
pac (numeric) – AC power [kW].
poa_global (numeric) – Total incident irradiance [W/m^2].
temp_air (numeric) – Ambient dry bulb temperature [C].
wind_speed (numeric) – Wind speed at a height of 10 meters [m/s].
*/

-- Join the pvdata table to the metrics table, mapping common_name values to needed metrics
-- Convert to correct units or drop rows with incorrect units (kWh, kVARh, mV)
CREATE OR REPLACE TABLE pvdata_2020_joined AS
SELECT 
    p.system_id,
    p.utc_measured_on,
    -- AC Power standardized strictly to kW
    MAX(CASE 
        WHEN m.common_name = 'AC power' AND LOWER(m.units) = 'w' THEN p.value / 1000.0
        WHEN m.common_name = 'AC power' AND LOWER(m.units) = 'kw' THEN p.value
        ELSE NULL -- Ignores kWh, kVARh, etc.
    END) AS ac_power_kw,
    -- POA Irradiance standardized strictly to W/m^2
    MAX(CASE 
        WHEN m.common_name = 'Irradiance POA' AND m.units = 'W/m^2' THEN p.value
        ELSE NULL -- Ignores uncalibrated mV readings
    END) AS poa_irradiance,
    -- Ambient Temp (already C)
    MAX(CASE WHEN m.common_name = 'Temperature ambient' THEN p.value END) AS ambient_temp,
    -- Wind Speed (already m/s)
    MAX(CASE WHEN m.common_name = 'Wind speed' THEN p.value END) AS wind_speed
FROM pvdata_2020_sample p
JOIN metrics m 
  ON p.system_id = m.system_id   -- choose matching system rows
 AND p.metric_id = m.metric_id   -- choose matching metric rows
GROUP BY 
    p.system_id,
    p.utc_measured_on
ORDER BY 
    system_id,
    utc_measured_on;

SELECT COUNT (*) FROM pvdata_2020_joined;
SELECT * FROM pvdata_2020_joined;
-- We now have a table with the following parameters:
--  Year: 2020 
--  Systems: 34, 35, 1200, 1201, 1202, 1239, 1276, 1277, 1278, 1283, 1367, 1418, 1419
--  2,483,851 rows of sensor data in chronological order, grouped by system
--  All systems have AC power, POA irradiance, ambient temperature, and wind speed present in the table, in the correct units
--  Schema: system_id, utc_measured_on, ac_power_kw, poa_irradiance, ambient_temp, wind_speed

-- Check AC power and POA irradiance correlation as sanity check
SELECT CORR(ac_power_kw, poa_irradiance) AS correlation_value
FROM pvdata_2020_joined;
-- It's 0.9204018588761711

-- Date range statistics per system
SELECT 
    system_id,
    MIN(utc_measured_on) AS start_time,
    MAX(utc_measured_on) AS end_time,
    COUNT(*) AS total_records,
    DATEDIFF(MAX(utc_measured_on), MIN(utc_measured_on)) AS total_days,
    ROUND(COUNT(*) / DATEDIFF(MAX(utc_measured_on), MIN(utc_measured_on)),2) AS avg_records_per_day,
    ROUND((COUNT(*) / DATEDIFF(MAX(utc_measured_on), MIN(utc_measured_on)))/1440,2) AS avg_records_per_minute
FROM pvdata_2020_joined
GROUP BY system_id
ORDER BY system_id;

-- AC Power summary statistics per system
-- Coerce nulls to 0? Meaning zero power production?
SELECT 
    system_id,
    ROUND(MIN(ac_power_kw),2) AS min_ac_power,
    ROUND(MAX(ac_power_kw),2) AS max_ac_power,
    ROUND(AVG(ac_power_kw),2) AS avg_ac_power,
    ROUND(STDDEV(ac_power_kw),2) AS std_ac_power,
    COUNT_IF(ac_power_kw IS NULL) AS ac_power_nulls,
    ROUND(COUNT_IF(ac_power_kw IS NULL) / COUNT(*),2) AS ac_power_null_percentage
FROM pvdata_2020_joined
GROUP BY system_id
ORDER BY system_id;

-- POA Irradiance summary statistics per system
SELECT 
    system_id,
    ROUND(MIN(poa_irradiance),2) AS min_poa_irradiance,
    ROUND(MAX(poa_irradiance),2) AS max_poa_irradiance,
    ROUND(AVG(poa_irradiance),2) AS avg_poa_irradiance,
    ROUND(STDDEV(poa_irradiance),2) AS std_poa_irradiance,
    COUNT_IF(poa_irradiance IS NULL) AS poa_irradiance_nulls,
    ROUND(COUNT_IF(poa_irradiance IS NULL) / COUNT(*),2) AS poa_irradiance_null_percentage
FROM pvdata_2020_joined
GROUP BY system_id
ORDER BY system_id;

-- Ambient temperature summary statistics per system
SELECT 
    system_id,
    ROUND(MIN(ambient_temp),2) AS min_ambient_temp,
    ROUND(MAX(ambient_temp),2) AS max_ambient_temp,
    ROUND(AVG(ambient_temp),2) AS avg_ambient_temp,
    ROUND(STDDEV(ambient_temp),2) AS std_ambient_temp,
    COUNT_IF(ambient_temp IS NULL) AS ambient_temp_nulls,
    ROUND(COUNT_IF(ambient_temp IS NULL) / COUNT(*),2) AS ambient_temp_null_percentage
FROM pvdata_2020_joined
GROUP BY system_id
ORDER BY system_id;

-- Wind speed summary statistics per system
SELECT 
    system_id,
    ROUND(MIN(wind_speed),2) AS min_wind_speed,
    ROUND(MAX(wind_speed),2) AS max_wind_speed,
    ROUND(AVG(wind_speed),2) AS avg_wind_speed,
    ROUND(STDDEV(wind_speed),2) AS std_wind_speed,
    COUNT_IF(wind_speed IS NULL) AS wind_speed_nulls,
    ROUND(COUNT_IF(wind_speed IS NULL) / COUNT(*),2) AS wind_speed_null_percentage
FROM pvdata_2020_joined
GROUP BY system_id
ORDER BY system_id;

-- Null percentages for all four metrics per system
CREATE TABLE IF NOT EXISTS system_null_percentages AS 
SELECT
    system_id,
    ROUND(COUNT_IF(ac_power_kw IS NULL) / COUNT(*),2) AS ac_power_null_percentage,
    ROUND(COUNT_IF(poa_irradiance IS NULL) / COUNT(*),2) AS poa_irradiance_null_percentage,
    ROUND(COUNT_IF(ambient_temp IS NULL) / COUNT(*),2) AS ambient_temp_null_percentage,
    ROUND(COUNT_IF(wind_speed IS NULL) / COUNT(*),2) AS wind_speed_null_percentage
FROM pvdata_2020_joined
GROUP BY system_id
ORDER BY system_id;

SELECT * FROM system_null_percentages;



In [0]:
# ==================================================================================
# Calculating total 2020 PR per system on raw data
# ==================================================================================

#### do this again in a new cell but with cleaned data

%pip install pvanalytics

import pvanalytics
from pvanalytics.metrics import performance_ratio_nrel
import pandas as pd
import pathlib
import matplotlib.pyplot as plt

# Load the Silver table into a PySpark DataFrame and convert to Pandas
df_spark = spark.table("pvdaq_catalog.silver.pvdata_2020_joined")
df = df_spark.toPandas()

# Ensure timestamp column is parsed and set as index
df['utc_measured_on'] = pd.to_datetime(df['utc_measured_on'])
df = df.set_index('utc_measured_on').sort_index()

# Define DC rated capacities (pdc0 in kW) per system
capacities_df = spark.table("pvdaq_catalog.silver.system").select("system_id", "power").toPandas() 

# Convert 'power' to numeric values (casts string numbers to floats and invalid entries to NaN)
capacities_df['power'] = pd.to_numeric(capacities_df['power'], errors='coerce')

# Calculate Performance Ratio per system over 2020
pr_results = {}

for system_id, group in df.groupby('system_id'):
    
    # Enforce mandatory DC capacity lookup
    if system_id not in system_dc_capacities or pd.isna(system_dc_capacities[system_id]):
        raise ValueError(
            f"Execution halted: system_id {system_id} does not have a mapped DC capacity (pdc0). "
            f"Please verify metadata in silver.system."
        )
    
    pdc0 = float(system_dc_capacities[system_id])

    # Calculate PR using required arguments: (poa_global, temp_air, wind_speed, pac, pdc0)
    pr = performance_ratio_nrel(
        poa_global=group['poa_irradiance'],
        temp_air=group['ambient_temp'],
        wind_speed=group['wind_speed'],
        pac=group['ac_power_kw'],
        pdc0=pdc0
    )
    
    pr_results[system_id] = pr

# Display results as a Pandas DataFrame
pr_summary = pd.DataFrame(list(pr_results.items()), columns=['system_id', 'performance_ratio_2020'])
print(pr_summary)


In [0]:
# ==================================================================================
# Plotting AC Power and Performance Ratio over 2020 for each system, raw data
# ==================================================================================


import pvanalytics
from pvanalytics.metrics import performance_ratio_nrel
import pandas as pd
import matplotlib.pyplot as plt

def plot_system_pr_and_power(system_id: int):
    """
    Loads data for a given system_id, calculates daily/annual NREL Performance Ratio,
    and plots PR alongside mean AC Power on a dual-axis time series chart.
    """
    # 1. Load data specifically for the provided system_id
    df_spark = spark.table("pvdaq_catalog.silver.pvdata_2020_joined").filter(f"system_id = {system_id}")
    df = df_spark.toPandas()

    if df.empty:
        raise ValueError(f"No time-series data found in pvdata_2020_joined for system_id {system_id}.")

    # Ensure timestamp column is parsed and set as index
    df['utc_measured_on'] = pd.to_datetime(df['utc_measured_on'])
    df = df.set_index('utc_measured_on').sort_index()

    # 2. Get DC capacity (pdc0) and normalize W to kW if necessary
    capacities_df = spark.table("pvdaq_catalog.silver.system").filter(f"system_id = {system_id}").select("power").toPandas()
    
    if capacities_df.empty or pd.isna(capacities_df['power'].iloc[0]):
        raise ValueError(f"Missing valid DC capacity metadata in silver.system for system_id {system_id}.")

    raw_power = float(capacities_df['power'].iloc[0])
    pdc0 = raw_power / 1000.0 if raw_power > 10000 else raw_power

    # 3. Calculate PR for the entire time series (for baseline horizontal line)
    daytime_full = df[df['poa_irradiance'] >= 50]
    if daytime_full.empty:
        raise ValueError(f"No daytime records (POA >= 50 W/m²) found for system_id {system_id}.")

    pr_whole_series = performance_ratio_nrel(
        poa_global=daytime_full['poa_irradiance'],
        temp_air=daytime_full['ambient_temp'],
        wind_speed=daytime_full['wind_speed'],
        pac=daytime_full['ac_power_kw'],
        pdc0=pdc0
    )

    # 4. Calculate Daily PR and aggregate Daily AC Power
    daily_metrics = []

    for date, data_subset in df.groupby(df.index.date):
        # Skip days with low/no irradiance to avoid zero-division noise
        daytime_data = data_subset[data_subset['poa_irradiance'] >= 50]
        if daytime_data.empty:
            continue

        # Calculate daily PR
        pr = performance_ratio_nrel(
            poa_global=daytime_data['poa_irradiance'],
            temp_air=daytime_data['ambient_temp'],
            wind_speed=daytime_data['wind_speed'],
            pac=daytime_data['ac_power_kw'],
            pdc0=pdc0
        )
        
        # Calculate total daily mean power output (kW)
        avg_ac_power = data_subset['ac_power_kw'].mean()
        
        daily_metrics.append({"date": date, "PR": pr, "avg_ac_power_kw": avg_ac_power})

    daily_df = pd.DataFrame(daily_metrics).set_index('date')

    # 5. Plot dual-axis time series
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Primary Axis (Left): NREL Performance Ratio
    color_pr = 'tab:blue'
    ax1.set_xlabel('Date')
    ax1.set_ylabel('NREL Performance Ratio', color=color_pr)
    line1 = ax1.plot(daily_df.index, daily_df['PR'], color=color_pr, label='Daily PR', alpha=0.8)
    line_base = ax1.axhline(pr_whole_series, color='red', linestyle='--', label=f'Annual PR Baseline ({pr_whole_series:.2f})')
    ax1.tick_params(axis='y', labelcolor=color_pr)

    # Secondary Axis (Right): AC Power Output (kW)
    ax2 = ax1.twinx()
    color_power = 'tab:orange'
    ax2.set_ylabel('Mean AC Power (kW)', color=color_power)
    line2 = ax2.plot(daily_df.index, daily_df['avg_ac_power_kw'], color=color_power, label='Mean Daily AC Power (kW)', alpha=0.5)
    ax2.tick_params(axis='y', labelcolor=color_power)

    # Combine legends from both axes
    lines = line1 + [line_base] + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')

    plt.title(f'System {system_id}: Daily NREL PR vs. Mean AC Power Output (2020)')
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()

plot_system_pr_and_power(1419)

In [0]:
# =============================================================================
# Time series data cleanup work with PVAnalytics
# =============================================================================

import pvanalytics
from pvanalytics.quality import gaps
import matplotlib.pyplot as plt
import pandas as pd
import pathlib
import matplotlib.dates as mdates

# Create individual tables of the joined time series data per system
system_ids = [34, 35, 1200, 1201, 1202, 1239, 1276, 1277, 1278, 1283, 1367, 1418, 1419]
for sys_id in system_ids:
    query = f"""
    CREATE OR REPLACE TABLE pvdata_2020_joined_{sys_id} AS
    SELECT *
    FROM pvdata_2020_joined
    WHERE system_id = {sys_id}
    """
    spark.sql(query)

# Identify interpolated data periods for AC power
df_spark = spark.table("pvdaq_catalog.silver.pvdata_2020_joined_1283")
df = df_spark.toPandas()

df['utc_measured_on'] = pd.to_datetime(df['utc_measured_on'])
df = df.set_index('utc_measured_on')

fig, ax = plt.subplots(figsize=(10, 5))

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
fig.autofmt_xdate()  # Rotates date labels so they don't overlap

detected_interpolated_data_mask = gaps.interpolation_diff(df['ac_power_kw'])

df['ac_power_kw'].plot()
df.loc[detected_interpolated_data_mask, "ac_power_kw"].plot(ls='', marker='.')

plt.legend(labels=["AC Power", "Detected Interpolated Data"])
plt.xlabel("Date")
plt.ylabel("Normalized AC Power")
plt.tight_layout()
plt.show()